# M04-P03 — Foundation Image Generation

En M04-P01 entrenamos pequeños generadores para entender VAE, GAN y Diffusion.

Ahora cambiamos de perspectiva:

```text
entrenar un modelo
        ↓
integrar un foundation model
```

Vamos a generar un activo visual empresarial real mediante **Amazon Bedrock** y a observar qué controlan el prompt, la semilla, el aspecto y el `negative_prompt`.

> Esta práctica genera **contenido sintético**. Las imágenes creadas aquí nunca deben tratarse como evidencia de una incidencia real.

## 1. Qué queremos demostrar

Durante la práctica:

1. haremos una llamada directa a `InvokeModel`;
2. decodificaremos la imagen devuelta en Base64;
3. compararemos un prompt simple con otro más específico usando la misma semilla;
4. cambiaremos la semilla para observar diversidad;
5. guardaremos el activo junto con metadatos de procedencia;
6. conectaremos esta capacidad con la futura superficie `Create` de Enterprise GenAI Assistant.

Usaremos el modelo configurado en el proyecto. Actualmente:

```text
Stable Image Core 1.0
stability.stable-image-core-v1:1
```

La generación visual se ejecuta en `us-west-2`, aunque el modelo de texto del curso siga utilizando `us-east-1`.

In [ ]:
from pathlib import Path
import base64
import io
import json
import time

import boto3
import matplotlib.pyplot as plt
from PIL import Image

from enterprise_genai.config import get_settings

settings = get_settings()

IMAGE_REGION = settings.bedrock_image_region
MODEL_ID = settings.bedrock_image_model_id

session = boto3.Session(
    profile_name=settings.aws_profile or None,
    region_name=IMAGE_REGION,
)
runtime = session.client("bedrock-runtime", region_name=IMAGE_REGION)
identity = session.client("sts", region_name=IMAGE_REGION).get_caller_identity()

OUT_DIR = Path("outputs/m04_p03")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("AWS identity:", identity["Arn"])
print("Image region:", IMAGE_REGION)
print("Image model:", MODEL_ID)

## 2. Primera llamada: abrir la caja

Los modelos de imagen de Stability AI se invocan con `InvokeModel` y un payload nativo del proveedor.

Nuestra primera petición será deliberadamente sencilla:

```text
prompt
aspect_ratio
seed
output_format
```

El servicio devuelve la imagen codificada en Base64. Vamos a realizar todos los pasos explícitamente antes de usar el helper reutilizable del proyecto.

In [ ]:
BASE_PROMPT = "A secure remote employee connecting a laptop to a corporate network, professional illustration"

seed = 42

payload = {
    "prompt": BASE_PROMPT,
    "aspect_ratio": "16:9",
    "seed": seed,
    "output_format": "png",
}

started = time.perf_counter()

response = runtime.invoke_model(
    modelId=MODEL_ID,
    body=json.dumps(payload),
    contentType="application/json",
    accept="application/json",
)

client_ms = (time.perf_counter() - started) * 1000

model_response = json.loads(response["body"].read().decode("utf-8"))

print("Request:")
print(json.dumps(payload, indent=2, ensure_ascii=False))
print()
print("Response fields:", list(model_response.keys()))
print("Seeds:", model_response.get("seeds"))
print("Finish reasons:", model_response.get("finish_reasons"))
print(f"Client time: {client_ms:.0f} ms")

In [ ]:
finish_reasons = model_response.get("finish_reasons", [])
finish_reason = finish_reasons[0] if finish_reasons else None

if finish_reason is not None:
    raise RuntimeError(f"La generación no terminó correctamente: {finish_reason}")

image_b64 = model_response["images"][0]
baseline_bytes = base64.b64decode(image_b64)
baseline_image = Image.open(io.BytesIO(baseline_bytes))

BASELINE_PATH = OUT_DIR / "baseline.png"
baseline_image.save(BASELINE_PATH)

display(baseline_image)

print("Imagen:", BASELINE_PATH)
print("Dimensiones:", baseline_image.size)
print("Bytes PNG:", len(baseline_bytes))

### Qué acabamos de hacer

La ruta real ha sido:

```text
prompt + parámetros
        ↓
InvokeModel
        ↓
JSON del proveedor
        ↓
imagen en Base64
        ↓
decode
        ↓
PNG
```

No hemos entrenado nada.

Ésta es una diferencia fundamental respecto a la GAN de M04-P01: estamos consumiendo una capacidad ya entrenada y pagando únicamente por la inferencia que solicitamos.

## 3. Prompt simple frente a prompt específico

Un prompt de imagen no tiene por qué ser largo, pero sí debe describir aquello que realmente necesitamos.

Vamos a mantener la **misma semilla** y comparar:

```text
prompt simple
vs
prompt específico
```

Así reducimos una fuente de variación y podemos observar mejor el efecto de las instrucciones.

In [ ]:
from enterprise_genai.image_generation import generate_image

DETAILED_PROMPT = (
    "Professional editorial illustration for an internal enterprise training course about secure remote access. "
    "A remote employee working on a laptop at a clean home-office desk, abstract visual metaphor of an encrypted "
    "VPN tunnel connecting to a modern corporate network operations center in the background. "
    "Clear composition, realistic lighting, polished corporate visual style, blue and neutral palette, "
    "wide cinematic framing, no readable text, no logos, no brand names."
)

detailed = generate_image(
    DETAILED_PROMPT,
    aspect_ratio="16:9",
    seed=42,
)

detailed_image = Image.open(io.BytesIO(detailed["image_bytes"]))
DETAILED_PATH = OUT_DIR / "detailed_seed_42.png"
detailed_image.save(DETAILED_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(baseline_image)
axes[0].set_title("Prompt simple · seed 42")
axes[0].axis("off")

axes[1].imshow(detailed_image)
axes[1].set_title("Prompt específico · seed 42")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("Latencia prompt específico:", detailed["latency_ms"], "ms")

### Qué debemos observar

No buscamos decidir cuál es “bonita”.

La pregunta útil es:

> ¿Cuál se aproxima mejor a un requisito empresarial concreto?

El prompt específico introduce información sobre:

- propósito;
- sujeto;
- contexto;
- composición;
- estilo;
- iluminación;
- paleta;
- relación de aspecto visual;
- elementos que no queremos.

En una aplicación real, el prompt forma parte de la lógica del producto.

## 4. La semilla controla una parte de la variación

Ahora mantenemos exactamente el mismo prompt y cambiamos únicamente:

```text
seed 42
→
seed 12345
```

La semilla no cambia el requisito. Cambia el punto de partida aleatorio de la generación.

In [ ]:
alternative = generate_image(
    DETAILED_PROMPT,
    aspect_ratio="16:9",
    seed=12345,
)

alternative_image = Image.open(io.BytesIO(alternative["image_bytes"]))
ALTERNATIVE_PATH = OUT_DIR / "detailed_seed_12345.png"
alternative_image.save(ALTERNATIVE_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(detailed_image)
axes[0].set_title("Mismo prompt · seed 42")
axes[0].axis("off")

axes[1].imshow(alternative_image)
axes[1].set_title("Mismo prompt · seed 12345")
axes[1].axis("off")

plt.tight_layout()
plt.show()

### Prompt y seed no significan lo mismo

```text
prompt
→ qué queremos

seed
→ una trayectoria concreta dentro del espacio de posibilidades
```

La semilla es útil para:

- volver a una variante concreta;
- comparar cambios de prompt con menos variación aleatoria;
- registrar cómo se produjo un activo.

No debemos confundir reproducibilidad de una configuración con garantía de que el servicio jamás cambie entre versiones futuras del modelo.

## 5. `negative_prompt`: especificar qué evitar

Además del prompt principal, Stable Image Core admite `negative_prompt`.

Lo usaremos para expresar defectos o elementos indeseados que son frecuentes en un activo de formación empresarial.

Esta llamada es útil para experimentar, pero no existe garantía de que un `negative_prompt` elimine absolutamente todo lo mencionado.

In [ ]:
NEGATIVE_PROMPT = (
    "readable text, captions, letters, logos, brand marks, watermark, low resolution, "
    "blurry image, distorted anatomy, duplicated objects, cluttered composition"
)

with_negative = generate_image(
    DETAILED_PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    aspect_ratio="16:9",
    seed=42,
)

negative_image = Image.open(io.BytesIO(with_negative["image_bytes"]))
NEGATIVE_PATH = OUT_DIR / "detailed_seed_42_negative_prompt.png"
negative_image.save(NEGATIVE_PATH)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(detailed_image)
axes[0].set_title("Sin negative_prompt")
axes[0].axis("off")

axes[1].imshow(negative_image)
axes[1].set_title("Con negative_prompt")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 6. Guardar procedencia junto al activo

Una imagen generada no debería quedar como un fichero sin contexto.

Vamos a conservar metadatos mínimos:

```text
modelo
región
prompt
negative_prompt
seed
aspect_ratio
latencia
```

Esto no convierte el activo en una evidencia real. Al contrario: ayuda a dejar claro que es **contenido sintético generado**.

In [ ]:
ASSET_PATH = OUT_DIR / "enterprise_remote_access_training.png"
METADATA_PATH = OUT_DIR / "enterprise_remote_access_training.json"

negative_image.save(ASSET_PATH)

metadata = {
    "synthetic": True,
    "purpose": "internal_training_asset",
    "model_id": with_negative["model_id"],
    "region": with_negative["region"],
    "prompt": DETAILED_PROMPT,
    "negative_prompt": NEGATIVE_PROMPT,
    "seed": with_negative["seed"],
    "aspect_ratio": with_negative["aspect_ratio"],
    "output_format": with_negative["output_format"],
    "latency_ms": with_negative["latency_ms"],
}

METADATA_PATH.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Activo:", ASSET_PATH)
print("Metadatos:", METADATA_PATH)
print()
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## 7. Entrenar frente a integrar

Después de M04-P01 podemos comparar las dos experiencias.

| Aspecto | GAN entrenada por nosotros | Foundation model |
|---|---|---|
| Dataset propio | Sí | No para esta práctica |
| Entrenamiento | Sí | Ya realizado por el proveedor |
| Hardware de entrenamiento | Nuestro problema | Abstraído |
| Resultado | Dígitos 28×28 | Activo visual complejo |
| Interfaz principal | PyTorch | API |
| Tiempo del desarrollador | Modelo + training loop | Prompt + integración |
| Control | Muy directo | Condicionado por la API/modelo |

La conclusión no es que entrenar sea inútil.

La conclusión es que para muchos productos el problema principal pasa a ser:

```text
definir el requisito
+
invocar el modelo
+
evaluar el resultado
+
integrarlo de forma segura
```

## 8. Un límite importante: generación no es evidencia

En M04-P02 una captura real era una fuente de evidencia.

Aquí sucede lo contrario:

```text
imagen generada
→ contenido sintético
→ NO evidencia
```

Una ilustración creada por `Create` puede utilizarse en:

- formación interna;
- documentación;
- artículos;
- prototipos;
- material visual.

Pero nunca debemos introducirla en una investigación como si mostrara algo que realmente ocurrió.

## 9. Modifica

Prueba **una** de estas modificaciones:

- cambia `seed`;
- cambia `aspect_ratio` a `1:1` o `9:16`;
- elimina una parte del prompt;
- cambia el estilo visual;
- modifica el `negative_prompt`.

Antes de ejecutar, predice qué esperas que cambie.

Después compara el resultado con la imagen anterior.

## 10. Siguiente paso

Ya tenemos un helper reutilizable:

```python
generate_image(...)
```

En M04-P04 lo incorporaremos a Enterprise GenAI Assistant como una nueva superficie:

```text
Create
```

La aplicación tendrá entonces dos capacidades visuales claramente separadas:

```text
Analyze
→ interpreta evidencia real

Create
→ produce contenido sintético
```

Mantener esa separación es una decisión funcional y de seguridad, no solo de interfaz.